# Probability, Confidence Intervals, and Hypothesis Testing

## Objectives

- Work with a normal probability distribution using SciPy
- Interpret density, cumulative probability, and percentile cutoffs
- Compare observations using standardized scores
- Construct and interpret a confidence interval for a population mean
- State null and alternative hypotheses
- Use one-sample, independent-sample, and paired *t* tests
- Make a decision using a significance level without overstating the conclusion



In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns

## 1. Probability distributions with `scipy`

SciPy provides probability-distribution objects with shared methods:

- `.rvs(size=...)`: generate random observations;
- `.pdf(x)`: evaluate the **density** of a continuous distribution at `x`;
- `.pmf(x)`: calculate the probability of a value for a discrete distribution;
- `.cdf(x)`: calculate the probability of observing a value less than or equal to `x`;
- `.ppf(p)`: find the value associated with cumulative probability `p`.

For a continuous distribution, the probability of observing one exact value is zero. The PDF height at one point is a density, not a probability. Probabilities come from intervals or cumulative areas.

### Guided example

Consider a normal distribution with mean 3 and standard deviation 5. We will write this as $(X \sim N(3, 5^2))$, where the second parameter is the variance if using the conventional $(N(\mu, \sigma^2))$ notation.

In [ ]:
# Create the distribution.
distribution = stats.norm(loc=3, scale=5)
distribution

In [ ]:
# Generate 20 reproducible random observations.
rng = np.random.default_rng(42)
sample = distribution.rvs(size=20, random_state=rng)
sample

In [ ]:
# Density at x = 4. This is not the probability of exactly 4.
density_at_4 = distribution.pdf(4)
print("Density at 4:", density_at_4)

In [ ]:
# Probability that X is 3 or less.
probability_below_3 = distribution.cdf(3)
print("P(X <= 3):", probability_below_3)

# Probability that X is between 0 and 6.
probability_between = distribution.cdf(6) - distribution.cdf(0)
print("P(0 <= X <= 6):", probability_between)

In [ ]:
# Plot the probability density.
x = np.linspace(distribution.ppf(0.001), distribution.ppf(0.999), 400)

plt.figure(figsize=(9, 4.5))
plt.plot(x, distribution.pdf(x))
plt.axvline(3, color="black", linestyle="--", label="Mean = 3")
plt.title("Normal Distribution with Mean 3 and Standard Deviation 5")
plt.xlabel("x")
plt.ylabel("Density")
plt.legend()

In [1]:
# Find and display the middle 90% of the distribution.
lower_cutoff = distribution.ppf(0.05)
upper_cutoff = distribution.ppf(0.95)

plt.figure(figsize=(9, 4.5))
plt.plot(x, distribution.pdf(x))
plt.fill_between(
    x,
    distribution.pdf(x),
    where=(x >= lower_cutoff) & (x <= upper_cutoff),
    alpha=0.35,
)
plt.axvline(lower_cutoff, color="black", linestyle="--")
plt.axvline(upper_cutoff, color="black", linestyle="--")
plt.title("Middle 90% of the Distribution")
plt.xlabel("x")
plt.ylabel("Density")

print("5th percentile:", round(lower_cutoff, 2))
print("95th percentile:", round(upper_cutoff, 2))

NameError: name 'distribution' is not defined

## 2. Standardization

Suppose scores come from two different distributions:

- English scores have mean 95 and standard deviation 5.
- Mathematics scores have mean 80 and standard deviation 7.

A standardized score measures how many standard deviations an observation lies above or below its distribution's mean:

$$z = \frac{x - \text{mean}}{\text{standard deviation}}$$

Standardization lets us compare relative position even when the original scales differ.

In [ ]:
#math class
math_class = stats.norm(loc = 80, scale = 7)

In [ ]:
#histogram
plt.hist(math_class.rvs(100))

In [ ]:
#english scores
english_class = stats.norm(loc = 95, scale = 5)

In [ ]:
#make a dataframe
tests_df = pd.DataFrame({'math': math_class.rvs(1000), 'english': english_class.rvs(1000)})
tests_df.head()

In [ ]:
#plot the histograms together
plt.hist(tests_df['math'])
plt.hist(tests_df['english'])
plt.grid();

### Quick problem

- Student A earned 82 in mathematics.
- Student B earned 97 in English.

Calculate both standardized scores. Which student performed better relative to their class distribution?

In [ ]:
# Student A: mathematics
student_a_z = (82 - 80) / 7
student_a_z

In [ ]:
# Student B: English
student_b_z = (97 - 95) / 5
student_b_z

In [ ]:
standardized_tests = (tests_df - tests_df.mean()) / tests_df.std()
standardized_tests.head()

### Practice problem 1 — Probability and percentiles

Suppose customer-service wait times are approximately normal with mean 12 minutes and standard deviation 3 minutes.

1. Create the distribution with `stats.norm`.
2. Find the probability of waiting 10 minutes or less.
3. Find the probability of waiting between 10 and 15 minutes.
4. Find the 90th percentile of wait time.
5. Plot the distribution and shade the area between 10 and 15 minutes.
6. Explain the 90th percentile in one sentence.

In [ ]:
# Practice problem 1
wait_times = stats.norm(loc=12, scale=3)

# Add your probability calculations here.

In [ ]:
# Plot and shade the requested interval here.

**Interpretation of the 90th percentile:**

>

## 3. Differences between groups

#### Differences between groups

In [ ]:
#read in the polls data
polls = pd.read_csv('https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/polls.csv')

In [ ]:
#take a peek
polls.head()

## 4. Confidence intervals

For a sample mean when the population standard deviation is unknown, a 95% confidence interval has the form

$$\bar{x} \pm t_{1 - \alpha / 2,\,n-1} \times \frac{s}{\sqrt{n}}.$$

The interval gives a range of plausible values for the population mean under the assumptions of the method. It does **not** mean that 95% of individual observations fall inside the interval.

- (\alpha): significance level chosen for the procedure; for 95% confidence, (\alpha=0.05)
- (t^*): critical value from the *t* distribution with (n-1) degrees of freedom
- (\bar{x}): sample mean
- (s): sample standard deviation
- (n): sample size
- (s/\sqrt{n}): standard error of the sample mean

In [ ]:
# A t distribution using the correct degrees of freedom for poll question 1.
q1 = polls["p1"].dropna()
q1_degrees_of_freedom = len(q1) - 1
t_dist = stats.t(df=q1_degrees_of_freedom)

print("Degrees of freedom:", q1_degrees_of_freedom)
print("Distribution mean:", t_dist.mean())

In [ ]:
x = np.linspace(-3, 3, 100)
plt.plot(x, t_dist.pdf(x))
plt.fill_between(x, t_dist.pdf(x), where = ((x > t_dist.ppf(.025)) &  (x< t_dist.ppf(.975))), hatch = '|', alpha = 0.5)
plt.grid()
plt.title('The t-distribution');
plt.xticks([-2, 2], ['2.5% of the data', '97.5% of the data']);

In [ ]:
# Examine the first poll-question data.
q1 = polls["p1"].dropna()
q1.head()

In [ ]:
#determine degrees of freedom
#i.e. length - 1
dof = len(q1) - 1
print(f'{dof} degrees of freedom')

In [ ]:
#look up test statistic
#we need our alpha and dof
#where do we bound 97.5% of our data
t_stat = stats.t.ppf(1 - 0.05/2, dof)
print(f'The t-statistic is {t_stat}')

In [ ]:
#compute sample standard deviation
s = np.std(q1, ddof = 1)
print(f'The sample standard deviation is {s}')

In [ ]:
#sample size
n = len(q1)
print(f'The sample size is {n}')

In [ ]:
#compute upper limit
upper = q1.mean() + t_stat*s/np.sqrt(n)
print(f'The upper limit of the confidence interval is {upper}')

In [ ]:
#compute the lower bound
lower = q1.mean() - t_stat*s/np.sqrt(n)
print(f'The lower limit of the confidence interval is {lower}')

In [ ]:
#print it
(lower, upper)

In [ ]:
#use scipy
#1 - alpha
#dof
#sem
#(1 - alpha, dof, mean, sem)
stats.t.interval(.95, n - 1, np.mean(q1), stats.sem(q1))

In [ ]:
# Simulate a sampling distribution of the mean.
sample_means = []

for sample_number in range(5000):
    one_sample_mean = q1.sample(20, replace=True).mean()
    sample_means.append(one_sample_mean)

sns.displot(sample_means, kind="kde")
plt.title("Sampling Distribution of the Mean")
plt.xlabel("Sample mean")
plt.show()

### Confidence-interval problem

1. Find the 95% confidence interval for the second poll question.
2. Compare the centers and widths of the two intervals.
3. Do the intervals overlap?
4. What can you reasonably say from this comparison?

**Caution:** overlap between two separate confidence intervals is a useful descriptive comparison, but it is not a formal hypothesis test of the difference between the means.

In [ ]:
q2 = polls["p2"].dropna()
q2.head()

### Jobs Data

The data below is a sample of job postings from New York City.  We want to investigate the lower and upper bound columns.

In [ ]:
#read in the data
jobs = pd.read_csv('https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/jobs.csv')

In [ ]:
#salary from
jobs.head()

### Margin of Error

Now, the question is to build a confidence interval that achieves a given amount of error.

$$error = z_{\alpha/2} \times \frac{\sigma}{\sqrt{n}}$$

**PROBLEM**

What is the minimum sample size necessary to estimate the upper salary range with 95% confidence within \$3000?

- need $z$-score: 1.96
- E: 3000
- $\sigma$: `np.std(jobs['salary_to'])`

In [ ]:
#do the computation


In [ ]:
#repeat for $500


## 6. Testing significance

A hypothesis test begins with two competing statements:

$$H_0: \text{null hypothesis}$$
$$H_a: \text{alternative hypothesis}$$

The null hypothesis generally represents no difference or a specified reference value. We choose a significance level, commonly (\alpha=0.05), before inspecting the result.

- If (p < \alpha), reject the null hypothesis.
- If (p \geq \alpha), fail to reject the null hypothesis.

“Fail to reject” does not prove that the null hypothesis is true. A *p*-value also does not measure the size or practical importance of an effect.

### Independent-samples example

The DRP data compare reading scores for a treatment group and a control group. For a two-sided test:

$$H_0: \mu_{treatment} = \mu_{control}$$

$$H_a: \mu_{treatment} \neq \mu_{control}$$

We will use Welch's independent-samples *t* test, which does not require the two groups to have equal variances.

In [ ]:
#read in the data
reading = pd.read_csv('https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/DRP.csv')
reading.head()

In [ ]:
#distributions of groups
sns.displot(x = 'drp', hue = 'group', data = reading, kind='kde')

For this test:

- Significance level: (\alpha = 0.05)
- Null hypothesis: the treatment and control population means are equal.
- Alternative hypothesis: the population means differ.

Before testing, inspect the group sizes, centers, spread, and distributions.

In [ ]:
# Split the groups using the labels documented in the dataset.
treatment = reading.loc[reading["group"] == "Treat", "drp"].dropna()
control = reading.loc[reading["group"] == "Control", "drp"].dropna()

print("Treatment observations:", len(treatment))
print("Control observations:", len(control))
print("Treatment mean:", treatment.mean())
print("Control mean:", control.mean())

In [ ]:
# Two-sided Welch t test.
two_sample_result = stats.ttest_ind(
    treatment,
    control,
    equal_var=False,
)

two_sample_result

In [ ]:
alpha = 0.05

if two_sample_result.pvalue < alpha:
    print("Reject the null hypothesis.")
else:
    print("Fail to reject the null hypothesis.")

**Interpret the result in context:**

Write one sentence reporting the sample means, *p*-value, and decision. Then write one sentence explaining what the test does **not** establish.

### A directional alternative

Suppose the question was specified in advance as whether the treatment produces a **higher** mean score:

$$H_0: \mu_{treatment} \leq \mu_{control}$$

$$H_a: \mu_{treatment} > \mu_{control}$$

SciPy can calculate the one-sided *p*-value directly. This is safer than automatically dividing a two-sided *p*-value by two.

In [ ]:
one_sided_result = stats.ttest_ind(
    treatment,
    control,
    equal_var=False,
    alternative="greater",
)

one_sided_result

In [ ]:
if one_sided_result.pvalue < alpha:
    print("Reject the null hypothesis in favor of a higher treatment mean.")
else:
    print("Fail to reject the null hypothesis.")

A directional alternative should be selected because it matches the research question—not because it produces a smaller *p*-value after looking at the data.

### Practice problem 2 — One-sample hypothesis test

A delivery service claims that its mean delivery time is 30 minutes. The following sample contains delivery times from 15 recent orders.

Test whether the evidence suggests that the mean delivery time is **greater than 30 minutes**.

1. State the null and alternative hypotheses.
2. Inspect the sample with a plot and numerical summary.
3. Use `stats.ttest_1samp(..., popmean=30, alternative="greater")`.
4. Make a decision using (\alpha=0.05).
5. Interpret the result in context without claiming that you proved the null hypothesis true or false.

In [ ]:
delivery_times = np.array([
    29, 31, 35, 28, 32,
    34, 33, 30, 36, 27,
    31, 34, 32, 33, 35,
])

# Inspect the sample and perform the one-sample test.

**Hypotheses, decision, and interpretation:**

>

### Practice problem 3 — Paired estimates

Insurance adjusters are concerned that estimates from Jocko's Garage are unusually high. The same 10 damaged cars were taken to Jocko's and to another garage.

Because each car produces a matched pair of estimates, this is a **paired** design.

1. Load and inspect the data.
2. Create a difference column: Jocko's estimate minus the other estimate.
3. Find the mean and standard deviation of the differences.
4. Plot the differences and check for unusual observations.
5. State hypotheses for whether Jocko's mean estimate is higher.
6. Use `stats.ttest_rel()` with the appropriate directional alternative.
7. Make and interpret a decision at (\alpha=0.05).

In [ ]:
jocko_url = "https://raw.githubusercontent.com/jfkoehler/nyu_bootcamp_fa25/refs/heads/main/data/JOCKO.csv"
jocko = pd.read_csv(jocko_url)
jocko.head()

In [ ]:
# Inspect column names, create the difference, and summarize it.

In [ ]:
# Run the paired t test and interpret the result.

# Group project presentation rehearsal

## Small-group practice — 15–18 minutes

Pair your project group with another project group. Each group should present from its current notebook or slides, even if the work is unfinished.

### Round 1 — Group A presents (4 minutes)

Your presentation should clearly identify:

1. the question and why it matters;
2. the dataset and unit of observation;
3. one important preparation or analytical decision;
4. the clearest visualization;
5. the main finding;
6. one limitation or unresolved question.

### Feedback to Group A (3 minutes)

The listening group should offer:

- one element that was especially clear;
- one claim that needs stronger evidence or explanation;
- one question a general audience might ask;
- one concrete revision to the visualization or presentation.

### Round 2 — Switch roles (7 minutes)

Group B presents for four minutes and receives three minutes of feedback.

### Revision (3–4 minutes)

Each group records the two most important changes it will make before the final presentation.

## Presentation feedback notes

### Feedback our group received

- **Clearest element:**
- **Claim needing more support:**
- **Audience question:**
- **Suggested revision:**

### Revisions we will make

1. 
2. 

### Presentation responsibility check

- Who introduces the question and data?
- Who explains the analysis or visualization?
- Who presents the conclusion and limitation?
- Can every member answer a question about the project?

# Final check for today

Before leaving, make sure you can explain:

1. the difference between a density and a probability;
2. what a 95% confidence interval is intended to estimate;
3. the roles of the null hypothesis, alternative hypothesis, and significance level;
4. how the *p*-value is used to make a decision;
5. why matched observations require a paired test;
6. one revision your group will make to its presentation.